In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
import torch.optim as optim
from torch.utils.data import DataLoader, random_split


train_transform=transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),
                        (0.5,0.5,0.5))
])

test_transform=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),
                        (0.5,0.5,0.5))
])


train_size=45000
val_size=5000
batch_size = 64


full_dataset = torchvision.datasets.CIFAR10(root=r"C:\Users\User\data", train=True, download=True, transform=train_transform)
test_dataset = torchvision.datasets.CIFAR10(root=r"C:\Users\User\data", train=False, download=True, transform=test_transform)

train_dataset,  val_dataset=random_split(
    full_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)
validation_full_dataset=torchvision.datasets.CIFAR10(root=r"C:\Users\User\data", train=True, download=True, transform=test_transform)

val_dataset.dataset=validation_full_dataset

trainloader=torch.utils.data.DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

valloader=torch.utils.data.DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)
testloader=torch.utils.data.DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))
print("Test images:", len(test_dataset))

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv_block1=nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )
        
        self.conv_block2=nn.Sequential(
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )

        self.conv_block3=nn.Sequential(
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )
        
        self.classifier=nn.Sequential(
            nn.Linear(128*4*4, 256),
            nn.ReLU(),
            nn.Dropout(0,5),
            nn.Linear(256, 10)
        )

    def forward(self, x):

        x=self.conv_block1(x)
        x=self.conv_block2(x)
        x=self.conv_block3(x)
        x=torch.flatten(x,1)
        x=self.classifier(x)
        return x





device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SimpleCNN().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 20
best_val_accuracy = 0.0

print(f"Training on {device}...")

for epoch in range(epochs):


    
    model.train()
    running_loss = 0.0

    for inputs, labels in trainloader:

        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    train_loss = running_loss / len(trainloader)


    
    model.eval()

    val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        for inputs, labels in valloader:

            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)

            loss = criterion(outputs, labels)

            val_loss += loss.item()

            _, predicted = torch.max(outputs, dim=1)

            total += labels.size(0)

            correct += (predicted == labels).sum().item()

    average_val_loss = val_loss / len(valloader)
    val_accuracy = 100 * correct / total

    print(
        f"Epoch {epoch + 1}/{epochs} | "
        f"Train Loss: {train_loss:.3f} | "
        f"Validation Loss: {average_val_loss:.3f} | "
        f"Validation Accuracy: {val_accuracy:.2f}%"
    )


    if val_accuracy > best_val_accuracy:

        best_val_accuracy = val_accuracy

        torch.save(
            model.state_dict(),
            "best_cnn_model.pth"
        )

print("Finished Training")
print(f"Best Validation Accuracy: {best_val_accuracy:.2f}%")


Training images: 45000
Validation images: 5000
Test images: 10000
Training on cpu...
Epoch 1/20 | Train Loss: 1.365 | Validation Loss: 1.006 | Validation Accuracy: 64.22%
Epoch 2/20 | Train Loss: 1.015 | Validation Loss: 0.849 | Validation Accuracy: 69.50%
Epoch 3/20 | Train Loss: 0.894 | Validation Loss: 0.891 | Validation Accuracy: 68.74%
Epoch 4/20 | Train Loss: 0.825 | Validation Loss: 0.791 | Validation Accuracy: 72.10%
Epoch 5/20 | Train Loss: 0.771 | Validation Loss: 0.721 | Validation Accuracy: 74.22%
Epoch 6/20 | Train Loss: 0.728 | Validation Loss: 0.695 | Validation Accuracy: 75.84%
Epoch 7/20 | Train Loss: 0.697 | Validation Loss: 0.717 | Validation Accuracy: 74.90%
Epoch 8/20 | Train Loss: 0.659 | Validation Loss: 0.664 | Validation Accuracy: 77.54%
Epoch 9/20 | Train Loss: 0.640 | Validation Loss: 0.625 | Validation Accuracy: 78.60%
Epoch 10/20 | Train Loss: 0.617 | Validation Loss: 0.604 | Validation Accuracy: 79.26%
Epoch 11/20 | Train Loss: 0.597 | Validation Loss: 0.5

In [4]:
# Test evaluation

model.eval()

correct = 0
total = 0

with torch.no_grad():

    for inputs, labels in testloader:

        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

test_accuracy = 100 * correct / total

print(f"Test Accuracy: {test_accuracy:.2f}%")

Test Accuracy: 82.44%
